In [ ]:
import os, sys, json, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 80)


In [ ]:
import os, sys, json, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 80)


In [ ]:
# ---- CHANGE THESE PATHS ----
BASELINE_JSON = Path("/scratch/network/cr7998/cv_emergence_project/results/probes/resnet50_cub_probes_attributes_fine.json")
CBM_JSON      = Path("/scratch/network/cr7998/cv_emergence_project/results/probes/resnet50_cub_cbm_probes_attributes_fine.json")

assert BASELINE_JSON.exists(), f"Missing baseline probe JSON: {BASELINE_JSON}"
assert CBM_JSON.exists(), f"Missing cbm probe JSON: {CBM_JSON}"

print("Baseline JSON:", BASELINE_JSON)
print("CBM JSON     :", CBM_JSON)


In [ ]:
def load_probe_json(path: Path, model_name: str) -> pd.DataFrame:
    with open(path, "r") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise TypeError(f"Expected list of dicts in {path}, got {type(data)}")
    df = pd.DataFrame(data)
    df["model"] = model_name
    return df

df_all = pd.concat(
    [load_probe_json(BASELINE_JSON, "baseline"),
     load_probe_json(CBM_JSON, "cbm")],
    ignore_index=True
)

required = {"layer", "target_type", "target_name", "best_val_acc", "group", "model"}
missing = required - set(df_all.columns)
assert not missing, f"JSON missing required columns: {missing}. Columns are: {list(df_all.columns)}"

print("Loaded df_all:", df_all.shape)
print("Models:", df_all["model"].unique())
print("Target types:", df_all["target_type"].unique())
df_all.head()


In [ ]:
_layer_pat = re.compile(r"^(layer[1-4])(?:\.(\d+))?$")

def _layer_sort_key(name: str):
    # order: conv1, layer1(.0..), layer2(.0..), layer3(.0..), layer4(.0..), avgpool
    if name == "conv1":
        return (0, 0, -1)
    if name == "avgpool":
        return (9, 0, -1)

    m = _layer_pat.match(name)
    if m:
        layer_block = int(m.group(1).replace("layer", ""))  # 1..4
        sub = m.group(2)
        sub_i = int(sub) if sub is not None else -1         # undotted layerX goes before layerX.0
        return (layer_block, 0, sub_i)

    # fallback: push unknowns to end but keep stable
    return (99, 0, 0)

LAYERS_ORDERED = sorted(df_all["layer"].unique().tolist(), key=_layer_sort_key)
layer_to_idx = {l: i for i, l in enumerate(LAYERS_ORDERED)}

print("Num layers inferred:", len(LAYERS_ORDERED))
print("First 25:", LAYERS_ORDERED[:25])
print("Last  15:", LAYERS_ORDERED[-15:])


In [ ]:
def get_curve(df: pd.DataFrame, target_type: str, target_name: str, model: str) -> pd.Series:
    sub = df[(df["model"] == model) &
             (df["target_type"] == target_type) &
             (df["target_name"] == target_name)].copy()
    if sub.empty:
        return pd.Series(index=LAYERS_ORDERED, dtype=float)

    # if duplicates, take max per layer
    sub = sub.groupby("layer", as_index=False)["best_val_acc"].max()
    sub["layer_idx"] = sub["layer"].map(layer_to_idx)
    sub = sub.sort_values("layer_idx")

    curve = pd.Series(sub["best_val_acc"].values, index=sub["layer"].values)
    return curve.reindex(LAYERS_ORDERED)  # may contain NaNs where that model has no measurement at that fine layer


In [ ]:
def sharp_rise_layer(curve: pd.Series) -> str | None:
    vals = curve.values.astype(float)
    if np.all(np.isnan(vals)):
        return None

    filled = pd.Series(vals).ffill().bfill().values
    diffs = np.diff(filled)
    if diffs.size == 0:
        return None

    j = int(np.argmax(diffs))
    return LAYERS_ORDERED[j + 1]  # later layer where jump is realized


In [ ]:
targets = df_all[["target_type", "target_name", "group"]].drop_duplicates()

rows = []
for _, t in targets.iterrows():
    ttype, tname, tgroup = t["target_type"], t["target_name"], t["group"]
    for model in ["baseline", "cbm"]:
        curve = get_curve(df_all, ttype, tname, model)
        if curve.empty or np.all(np.isnan(curve.values)):
            continue
        rows.append({
            "model": model,
            "target_type": ttype,
            "target_name": tname,
            "group": tgroup,
            "max_acc": float(np.nanmax(curve.values)),
            "emergence_layer": sharp_rise_layer(curve),
        })

emerge_df = pd.DataFrame(rows)
emerge_df["layer_idx"] = emerge_df["emergence_layer"].map(layer_to_idx)

print("emerge_df rows:", len(emerge_df))
emerge_df.head()


In [ ]:
# Keep attributes only (not species) unless you want species too
attr_em = emerge_df[emerge_df["target_type"] == "attribute"].copy()

wide = attr_em.pivot_table(
    index=["target_name", "group"],
    columns="model",
    values=["layer_idx", "emergence_layer", "max_acc"],
    aggfunc="first",
)

# Flatten columns
wide.columns = ["_".join(col).strip() for col in wide.columns.values]
wide = wide.reset_index()

# Compute changes
wide["delta_layer_idx"] = wide["layer_idx_cbm"] - wide["layer_idx_baseline"]
wide["delta_max_acc"]   = wide["max_acc_cbm"]   - wide["max_acc_baseline"]

# Basic counts
n_total = len(wide)
n_changed = int((wide["delta_layer_idx"] != 0).sum())
n_earlier = int((wide["delta_layer_idx"] < 0).sum())
n_later   = int((wide["delta_layer_idx"] > 0).sum())
n_same    = int((wide["delta_layer_idx"] == 0).sum())

print(f"Total attributes: {n_total}")
print(f"Changed emergence layer: {n_changed} ({n_changed/n_total:.2%})")
print(f"  Earlier under CBM (negative shift): {n_earlier}")
print(f"  Later under CBM (positive shift):   {n_later}")
print(f"  Same layer:                          {n_same}")

# Show some of the biggest shifts
show_cols = [
    "target_name","group",
    "emergence_layer_baseline","emergence_layer_cbm",
    "delta_layer_idx",
    "max_acc_baseline","max_acc_cbm","delta_max_acc"
]

print("\nTop 15 shifts to LATER (CBM deeper):")
display(wide.sort_values("delta_layer_idx", ascending=False)[show_cols].head(15))

print("\nTop 15 shifts to EARLIER (CBM earlier):")
display(wide.sort_values("delta_layer_idx", ascending=True)[show_cols].head(15))


In [ ]:
def biggest_jump(curve: pd.Series) -> float:
    v = curve.values.astype(float)
    if np.all(np.isnan(v)):
        return np.nan
    filled = pd.Series(v).ffill().bfill().values
    d = np.diff(filled)
    return float(np.nanmax(d)) if d.size else np.nan

def plot_attr_curve(attr_name: str):
    c0 = get_curve(df_all, "attribute", attr_name, "baseline")
    c1 = get_curve(df_all, "attribute", attr_name, "cbm")
    plot_curve_fine_connected(c0, c1, f"Attribute probe accuracy vs depth: {attr_name}", LAYERS_ORDERED)

# Add jump stats for selection
jump_rows = []
for _, r in wide.iterrows():
    a = r["target_name"]
    c0 = get_curve(df_all, "attribute", a, "baseline")
    c1 = get_curve(df_all, "attribute", a, "cbm")
    jump_rows.append({
        "target_name": a,
        "jump_baseline": biggest_jump(c0),
        "jump_cbm": biggest_jump(c1),
        "delta_jump": biggest_jump(c1) - biggest_jump(c0),
    })
jump_df = pd.DataFrame(jump_rows)

wide2 = wide.merge(jump_df, on="target_name", how="left")
wide2["abs_delta_layer"] = wide2["delta_layer_idx"].abs()
wide2["abs_delta_acc"]   = wide2["delta_max_acc"].abs()
wide2["abs_delta_jump"]  = wide2["delta_jump"].abs()

# Optional: filter to "reliable" attributes
MIN_MAXACC = 0.70
reliable = wide2[(wide2["max_acc_baseline"] >= MIN_MAXACC) | (wide2["max_acc_cbm"] >= MIN_MAXACC)].copy()
print("Reliable attributes:", len(reliable), "of", len(wide2))

# Pick top-k by emergence shift (tie-break by acc change)
TOPK = 6
picked = (reliable
          .sort_values(["abs_delta_layer","abs_delta_acc"], ascending=False)
          .head(TOPK)["target_name"]
          .tolist())

print("Picked attributes (largest emergence-layer shifts):")
for a in picked:
    row = reliable[reliable["target_name"] == a].iloc[0]
    print(f"  - {a} | {row['emergence_layer_baseline']} -> {row['emergence_layer_cbm']} (Δidx={row['delta_layer_idx']}) | Δacc={row['delta_max_acc']:.3f}")

for a in picked:
    plot_attr_curve(a)


In [ ]:
# by accuracy change
#picked = reliable.sort_values("abs_delta_acc", ascending=False).head(TOPK)["target_name"].tolist()

# by jump change
#picked = reliable.sort_values("abs_delta_jump", ascending=False).head(TOPK)["target_name"].tolist()


In [ ]:
g = (wide2.groupby("group")
     .agg(
         n=("target_name","count"),
         mean_base=("layer_idx_baseline","mean"),
         mean_cbm=("layer_idx_cbm","mean"),
         mean_shift=("delta_layer_idx","mean"),
         frac_changed=("delta_layer_idx", lambda x: (x != 0).mean()),
     )
     .reset_index()
     .sort_values("mean_shift", ascending=False))

display(g.head(20))

TOPG = 12
top = g.sort_values("mean_shift", ascending=False).head(TOPG)

plt.figure(figsize=(12,5))
x = np.arange(len(top))
plt.bar(x-0.2, top["mean_base"].values, width=0.4, label="Baseline")
plt.bar(x+0.2, top["mean_cbm"].values,  width=0.4, label="CBM")
plt.xticks(x, top["group"].values, rotation=45, ha="right")
plt.ylabel("Mean emergence depth index")
plt.title("Groups with largest increase in mean emergence depth under CBM (fine-grained)")
plt.legend()
plt.tight_layout()
plt.show()
